# Phase 3: LLM Cluster Labeling (Venhoff et al. 2025 Recipe)

**Input**: cluster_data_n15.json from Phase 2 (top-100 + random-100 sentences per cluster).

**Goal**: label each of the 15 clusters with {title, description} via GPT-4o-mini, following Appendix B.1 Prompt 1 of the paper.

**Output**: `cluster_labels.json` — {cluster_id → {title, description}} — uploaded to HF.

**Cost**: ~$0.50 via GPT-4o-mini. **Time**: ~5 min.

**Setup**: need `OPENAI_API_KEY` as Colab secret.

## Cell 1 — Setup

In [ ]:
import sys, subprocess
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)
try:
    import openai, huggingface_hub
except Exception:
    pip('install', '-q', 'openai', 'huggingface_hub==1.5.0', 'hf_transfer')

import json, time, os
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi, login

# Auth
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: login(token=hf_token); print('HF auth OK')
    openai_key = userdata.get('OPENAI_API_KEY')
    if not openai_key: raise RuntimeError('missing OPENAI_API_KEY Colab secret')
    os.environ['OPENAI_API_KEY'] = openai_key
    print('OpenAI auth OK')
except Exception as e:
    print(f'auth issue: {e}')

from openai import OpenAI
client = OpenAI()

OUT = Path('/content/cluster_labels'); OUT.mkdir(exist_ok=True)
HF_REPO = 'caiovicentino1/qwen35-a3b-sae-phase2'
N_LATENTS = 15

## Cell 2 — Load cluster data from HF

In [ ]:
path = snapshot_download(HF_REPO, repo_type='model', cache_dir='/content/cache')
with open(Path(path) / f'cluster_data_n{N_LATENTS}.json') as f:
    cluster_data = json.load(f)
print(f'Clusters: {len(cluster_data)}')
for cid, cd in list(cluster_data.items())[:3]:
    print(f'\nCluster {cid}: {cd["size"]} sentences')
    print(f'  sample top: {cd["top100"][0][:100]}')

## Cell 3 — Label each cluster via GPT-4o-mini

In [ ]:
# Prompt adapted from Venhoff et al. Appendix B.1 Prompt 1
SYSTEM = """You are analyzing clusters of sentences extracted from a reasoning language model's chain-of-thought traces. Each cluster represents a specific cognitive function the model performs during reasoning. Your task is to identify and describe the cognitive function that characterizes a given cluster."""

USER_TEMPLATE = """I have a cluster of sentences that co-activate a specific sparse autoencoder feature during the model's reasoning process. Below are 30 sentences highly representative of this cluster (top-activating), followed by 20 randomly-sampled sentences also from this cluster.

Top-activating (30):
{top_sentences}

Random samples (20):
{random_sentences}

Based on these sentences, identify the core cognitive function this cluster represents. Return a JSON object:

{{
  "title": "<2-5 word title in Title Case, e.g. 'Recalling Formulas', 'Verifying Intermediate Steps'>",
  "description": "<3-4 sentence description of the cognitive function, what triggers it, and how the model uses it during reasoning>",
  "confidence": "<high|medium|low — how coherent is this cluster>"
}}

Respond ONLY with the JSON object, no additional text."""

labels = {}
t0 = time.time()
total_cost = 0.0
# GPT-4o-mini pricing Apr 2026: $0.15/M input, $0.60/M output

for cid in sorted(cluster_data.keys(), key=int):
    cd = cluster_data[cid]
    if cd['size'] == 0:
        labels[cid] = {'title': 'Empty Cluster', 'description': 'No sentences in this cluster', 'confidence': 'none'}
        continue
    # Take 30 top + 20 random (capped at available)
    top_s = cd['top100'][:30]
    rand_s = cd['random100'][:20]
    top_str = '\n'.join(f'- {s[:250]}' for s in top_s)
    rand_str = '\n'.join(f'- {s[:250]}' for s in rand_s)
    prompt = USER_TEMPLATE.format(top_sentences=top_str, random_sentences=rand_str)
    try:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role':'system','content':SYSTEM},
                      {'role':'user','content':prompt}],
            temperature=0.2,
            response_format={'type':'json_object'},
            max_tokens=400,
        )
        text = resp.choices[0].message.content
        label = json.loads(text)
        labels[cid] = label
        usage = resp.usage
        cost = (usage.prompt_tokens * 0.15 + usage.completion_tokens * 0.60) / 1e6
        total_cost += cost
        print(f'Cluster {cid}: {label["title"]} ({label["confidence"]})')
        print(f'  {label["description"][:150]}...')
    except Exception as e:
        print(f'Cluster {cid} FAILED: {e}')
        labels[cid] = {'title': 'Error', 'description': str(e)[:200], 'confidence': 'none'}

print(f'\n✅ {len(labels)} clusters labeled in {(time.time()-t0):.1f}s')
print(f'Total cost: ${total_cost:.3f}')
with open(OUT / f'cluster_labels_n{N_LATENTS}.json', 'w') as f:
    json.dump(labels, f, indent=2, ensure_ascii=False)

## Cell 4 — Review + upload

In [ ]:
print('=== 15 reasoning categories (Qwen3.5-35B-A3B) ===\n')
for cid in sorted(labels.keys(), key=int):
    l = labels[cid]
    marker = {'high':'⭐','medium':'✓','low':'?','none':'✗'}.get(l.get('confidence','none'), '?')
    print(f'{marker} Cluster {cid}: {l["title"]}')
    print(f'   {l["description"]}')
    print()

# Confidence distribution
from collections import Counter
conf = Counter(l.get('confidence','none') for l in labels.values())
print(f'Confidence: {dict(conf)}')

# Upload
api = HfApi()
api.upload_file(path_or_fileobj=str(OUT / f'cluster_labels_n{N_LATENTS}.json'),
                path_in_repo=f'cluster_labels_n{N_LATENTS}.json',
                repo_id=HF_REPO, repo_type='model',
                commit_message=f'Phase 3: GPT-4o-mini cluster labels for n={N_LATENTS}')
print(f'\n✅ uploaded to https://huggingface.co/{HF_REPO}')